In [ ]:
import pandas as pd
import numpy as np

# Load the BTS dataset
print('Loading BTS flight data...')
df = pd.read_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/raw/bts_flights_2025.csv', low_memory=False)

# Always start every new dataset with these four commands
print('Shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())
df.head()

# Filter to the six major U.S. carriers
major_carriers = ['UA', 'DL', 'AA', 'WN', 'AS', 'B6']
df = df[df['OP_UNIQUE_CARRIER'].isin(major_carriers)].copy()
print(f'Rows after filtering to major carriers: {len(df):,}')

# Map carrier codes to full airline names
airline_names = {
    'UA': 'United Airlines',
    'DL': 'Delta Air Lines',
    'AA': 'American Airlines',
    'WN': 'Southwest Airlines',
    'AS': 'Alaska Airlines',
    'B6': 'JetBlue Airways'
}
df['airline_name'] = df['OP_UNIQUE_CARRIER'].map(airline_names)

# Fix the flight date column
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], errors='coerce')

# Extract time components
df['month']       = df['FL_DATE'].dt.month
df['month_name']  = df['FL_DATE'].dt.strftime('%B')
df['day_of_week'] = df['FL_DATE'].dt.day_name()
df['quarter']     = df['FL_DATE'].dt.quarter

print('Date range:', df['FL_DATE'].min(), 'to', df['FL_DATE'].max())
print('Flights per airline:')
print(df['airline_name'].value_counts())

# Handle cancelled flights
df_cancelled = df[df['CANCELLED'] == 1].copy()
df_operated  = df[df['CANCELLED'] == 0].copy()

print(f'Total flights: {len(df):,}')
print(f'Operated: {len(df_operated):,} ({len(df_operated)/len(df)*100:.1f}%)')
print(f'Cancelled: {len(df_cancelled):,} ({len(df_cancelled)/len(df)*100:.1f}%)')

# Fill NaN delay causes with 0
delay_cols = ['CARRIER_DELAY','WEATHER_DELAY','NAS_DELAY','SECURITY_DELAY','LATE_AIRCRAFT_DELAY']
df_operated = df_operated.copy()
df_operated[delay_cols] = df_operated[delay_cols].fillna(0)

# Fill ArrDelay and DepDelay nulls with 0
df_operated['ARR_DELAY'] = df_operated['ARR_DELAY'].fillna(0)
df_operated['DEP_DELAY'] = df_operated['DEP_DELAY'].fillna(0)

print('\nRemaining nulls in operated flights:')
print(df_operated[['ARR_DELAY','DEP_DELAY']+delay_cols].isnull().sum())

# On-time flag
df_operated['on_time'] = df_operated['ARR_DELAY'] <= 15

# Delay severity category
def delay_severity(mins):
    if mins <= 15:    return 'On Time'
    elif mins <= 45:  return 'Minor Delay'
    elif mins <= 120: return 'Moderate Delay'
    elif mins <= 240: return 'Severe Delay'
    else:             return 'Extreme Delay (4hr+)'

df_operated['delay_severity'] = df_operated['ARR_DELAY'].apply(delay_severity)

# Primary delay cause
def primary_cause(row):
    causes = {
        'Carrier':       row['CARRIER_DELAY'],
        'Weather':       row['WEATHER_DELAY'],
        'NAS':           row['NAS_DELAY'],
        'Security':      row['SECURITY_DELAY'],
        'Late Aircraft': row['LATE_AIRCRAFT_DELAY']
    }
    max_cause = max(causes, key=causes.get)
    return max_cause if causes[max_cause] > 0 else 'No Delay'

df_operated['primary_delay_cause'] = df_operated.apply(primary_cause, axis=1)

# Distance tier
df_operated['distance_tier'] = pd.cut(
    df_operated['DISTANCE'],
    bins=[0, 500, 1000, 2000, 9999],
    labels=['Short (<500mi)','Medium (500-1000mi)','Long (1000-2000mi)','Ultra Long (2000mi+)']
)

print('Business metrics created successfully')
print('On-time rate:', round(df_operated['on_time'].mean()*100, 2), '%')
print('\nDelay severity breakdown:')
print(df_operated['delay_severity'].value_counts())

# Export cleaned data
df_operated.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/cleaned/flights_cleaned.csv', index=False)
print(f'Operated flights saved: {len(df_operated):,} rows, {len(df_operated.columns)} columns')

df_cancelled.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/cleaned/flights_cancelled.csv', index=False)
print(f'Cancelled flights saved: {len(df_cancelled):,} rows')

print(f'\nCleaning Summary:')
print(f'Original rows:  {len(df):,}')
print(f'Operated rows:  {len(df_operated):,}')
print(f'Cancelled rows: {len(df_cancelled):,}')
print(f'Carriers: {df_operated["airline_name"].nunique()}')
print(f'Airports: {df_operated["ORIGIN"].nunique()}')

Loading BTS flight data...
Shape: (7001619, 16)

Column names:
['FL_DATE', 'OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

Data types:
FL_DATE                    str
OP_UNIQUE_CARRIER          str
OP_CARRIER_FL_NUM        int64
ORIGIN                     str
DEST                       str
DEP_DELAY              float64
ARR_DELAY              float64
CANCELLED              float64
CANCELLATION_CODE          str
AIR_TIME               float64
DISTANCE               float64
CARRIER_DELAY          float64
WEATHER_DELAY          float64
NAS_DELAY              float64
SECURITY_DELAY         float64
LATE_AIRCRAFT_DELAY    float64
dtype: object

Missing values per column:
FL_DATE                      0
OP_UNIQUE_CARRIER            0
OP_CARRIER_FL_NUM            0
ORIGIN                       0
DEST              

/var/folders/r9/t17b_5hs6735lcmv989v1nzm0000gq/T/ipykernel_64454/733434000.py:35: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], errors='coerce')


Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
Flights per airline:
airline_name
Southwest Airlines    1391885
Delta Air Lines       1026332
American Airlines      973653
United Airlines        795271
Alaska Airlines        245588
JetBlue Airways        231413
Name: count, dtype: int64
Total flights: 4,664,142
Operated: 4,610,224 (98.8%)
Cancelled: 53,918 (1.2%)

Remaining nulls in operated flights:
ARR_DELAY              0
DEP_DELAY              0
CARRIER_DELAY          0
WEATHER_DELAY          0
NAS_DELAY              0
SECURITY_DELAY         0
LATE_AIRCRAFT_DELAY    0
dtype: int64
Business metrics created successfully
On-time rate: 78.5 %

Delay severity breakdown:
delay_severity
On Time                 3618866
Minor Delay              522637
Moderate Delay           326477
Severe Delay             105921
Extreme Delay (4hr+)      36323
Name: count, dtype: int64
Operated flights saved: 4,610,224 rows, 25 columns
Cancelled flights saved: 53,918 rows

Cleaning Summary:
Origina